# 自动化 RAG 配置调优 Loop

告别繁琐的手动微调，构建一个**自主搜索配置、评估召回率、达标即止**的自动化 Loop（循环）系统。
![RAG Loop 动画](/mnt/d/GLNYYGL/RAG_langchain/data/rag_loop_parameter_search.gif)
---

## 💡 为什么用 Loop 调优 RAG？

手动调优（反复修改 Chunk、Embedding、Reranker 并手动跑分）极其繁琐且易混淆。RAG 调优天然适合 Loop 处理：

* **客观自动裁决**：评估集上的 Recall@k 是确定性指标，无需人工干预。
* **高效搜索**：Loop 能系统化遍历配置空间，准确记录历史方案与结果。

---

## 🛠️ 准备哪些

在启动 Loop 前，需确保准备好以下模块：

1. **参数化 RAG 管道**：支持动态修改 Chunk 大小/重叠度、Embedding 模型、候选数 $k$、Reranker 开关等。
2. **评估数据集**：30–50 个带有标注正确源 Chunk 的测试问题。
3. **评估函数（Oracle）**：能自动运行测试并输出 Recall@k 的判定函数。

In [ ]:
%pip install -q "sentence-transformers>=5.4.0" datasets openai pandas rank-bm25

In [1]:
import os, json
from getpass import getpass
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from openai import OpenAI
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EMBEDDING_MODELS = {
    "Qwen3-Embedding-0.6B": "../Qwen/Qwen3-Embedding-0.6B",
}
RERANKER_MODELS = {
    "Qwen3-Reranker-0.6B": "../Qwen/Qwen3-Reranker-0.6B",
}

SEARCH_SPACE = {
    "chunk_size": [400, 600, 800, 1200],
    "chunk_overlap": [0, 100, 200],
    ##"embedding": ["Qwen3-Embedding-0.6B" ,"text-embedding-3-small", "bge-large", "e5-large"],
    "embedding": ["Qwen3-Embedding-0.6B"],
    "k": [5, 10, 20],
    ## "reranker": [None, "bge-reranker", "cohere-rerank"],
    "reranker": [None, "Qwen3-Reranker-0.6B"],
    "hybrid": [False, True],
}

INITIAL_CONFIG = {
    "chunk_size": 400,
    "chunk_overlap": 0,
    "embedding": "Qwen3-Embedding-0.6B",
    "k": 5,
    "reranker": None,
    "hybrid": False,
}

FINAL_K = 5
TARGET_RECALL = 0.90
MAX_ROUNDS = 6

TRAIN_QUERIES = 30
HOLDOUT_QUERIES = 15
CORPUS_SIZE = 200

if not os.getenv("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = getpass("DEEPSEEK_API_KEY: ")

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

print("Device:", DEVICE)
print("Search space:", SEARCH_SPACE)

Device: cuda
Search space: {'chunk_size': [400, 600, 800, 1200], 'chunk_overlap': [0, 100, 200], 'embedding': ['Qwen3-Embedding-0.6B'], 'k': [5, 10, 20], 'reranker': [None, 'Qwen3-Reranker-0.6B'], 'hybrid': [False, True]}


## 1. 加载 SciFact 真实数据

选取 30 条查询供 Loop 调参，15 条查询做 held-out 验证。检索库保留这些查询的所有正确论文，再加入真实干扰论文。

In [2]:
corpus_ds = load_dataset("BeIR/scifact", "corpus", split="corpus")
queries_ds = load_dataset("BeIR/scifact", "queries", split="queries")
qrels_ds = load_dataset("BeIR/scifact-qrels", split="test")

all_corpus = {
    str(x["_id"]): {
        "title": x["title"],
        "text": x["text"],
    }
    for x in corpus_ds
}
queries = {str(x["_id"]): x["text"] for x in queries_ds}

qrels = defaultdict(set)
for x in qrels_ds:
    qrels[str(x["query-id"])].add(str(x["corpus-id"]))

rng = np.random.default_rng(42)
query_ids = np.array(sorted(qrels))
rng.shuffle(query_ids)

train_ids = query_ids[:TRAIN_QUERIES].tolist()
holdout_ids = query_ids[TRAIN_QUERIES:TRAIN_QUERIES + HOLDOUT_QUERIES].tolist()
eval_ids = train_ids + holdout_ids

gold_doc_ids = set().union(*(qrels[qid] for qid in eval_ids))
other_doc_ids = [x for x in all_corpus if x not in gold_doc_ids]
rng.shuffle(other_doc_ids)

selected_ids = list(gold_doc_ids) + other_doc_ids[:CORPUS_SIZE - len(gold_doc_ids)]
corpus = {doc_id: all_corpus[doc_id] for doc_id in selected_ids}

print(f"Corpus: {len(corpus):,} documents")
print(f"Train: {len(train_ids)} | Holdout: {len(holdout_ids)}")

Corpus: 1,200 documents
Train: 30 | Holdout: 15


## 2. 根据配置建立真实检索索引

每组 `chunk_size + chunk_overlap + embedding` 都会建立对应的 chunk 索引。  
索引包含：

- Qwen3-Embedding 生成的真实向量；
- BM25 使用的词法索引；
- chunk 到原始论文的映射。

In [ ]:
EMBEDDER_CACHE = {}
RERANKER_CACHE = {}
INDEX_CACHE = {}
QUERY_VECTOR_CACHE = {}

def get_embedder(name):
    if name not in EMBEDDER_CACHE:
        EMBEDDER_CACHE[name] = SentenceTransformer(
            EMBEDDING_MODELS[name],
            device=DEVICE,
        )
    return EMBEDDER_CACHE[name]

def get_reranker(name):
    if name not in RERANKER_CACHE:
        RERANKER_CACHE[name] = CrossEncoder(
            RERANKER_MODELS[name],
            device=DEVICE,
        )
    return RERANKER_CACHE[name]

def split_text(text, chunk_size, chunk_overlap):
    step = chunk_size - chunk_overlap
    return [
        text[start:start + chunk_size]
        for start in range(0, len(text), step)
        if text[start:start + chunk_size].strip()
    ]

def build_index(config):
    key = (
        config["chunk_size"],
        config["chunk_overlap"],
        config["embedding"],
    )
    if key in INDEX_CACHE:
        return INDEX_CACHE[key]

    chunk_texts, chunk_doc_ids = [], []
    for doc_id, doc in corpus.items():
        full_text = f'{doc["title"]}\n{doc["text"]}'
        for chunk in split_text(
            full_text,
            config["chunk_size"],
            config["chunk_overlap"],
        ):
            chunk_texts.append(chunk)
            chunk_doc_ids.append(doc_id)

    embedder = get_embedder(config["embedding"])
    dense_vectors = embedder.encode(
        chunk_texts,
        normalize_embeddings=True,
        batch_size=16,
        show_progress_bar=True,
    )

    bm25 = BM25Okapi([text.lower().split() for text in chunk_texts])

    index = {
        "texts": chunk_texts,
        "doc_ids": chunk_doc_ids,
        "dense_vectors": np.asarray(dense_vectors),
        "bm25": bm25,
    }
    INDEX_CACHE[key] = index
    return index

def get_query_vector(qid, embedding_name):
    key = (qid, embedding_name)
    if key not in QUERY_VECTOR_CACHE:
        QUERY_VECTOR_CACHE[key] = get_embedder(embedding_name).encode(
            queries[qid],
            prompt_name="query",
            normalize_embeddings=True,
        )
    return QUERY_VECTOR_CACHE[key]

## 3. 执行当前配置

- `hybrid=False`：只使用 Qwen3-Embedding；
- `hybrid=True`：使用 Reciprocal Rank Fusion 融合 Dense 与 BM25；
- `k`：第一阶段保留的候选 chunk 数；
- `reranker=None`：直接使用第一阶段排序；
- 指定 Qwen3-Reranker：真实重排候选 chunk；
- 最终统一去重为前 5 篇论文并计算 Recall@5。

In [ ]:
def first_stage_rank(qid, config, index):
    query_vector = get_query_vector(qid, config["embedding"])
    dense_scores = index["dense_vectors"] @ query_vector
    dense_order = np.argsort(-dense_scores)

    if not config["hybrid"]:
        return dense_order[:config["k"]].tolist()

    bm25_scores = index["bm25"].get_scores(queries[qid].lower().split())
    bm25_order = np.argsort(-bm25_scores)

    # Reciprocal Rank Fusion
    pool_size = min(max(config["k"] * 5, 50), len(dense_order))
    fused = defaultdict(float)

    for rank, idx in enumerate(dense_order[:pool_size]):
        fused[int(idx)] += 1.0 / (60 + rank + 1)

    for rank, idx in enumerate(bm25_order[:pool_size]):
        fused[int(idx)] += 1.0 / (60 + rank + 1)

    return [
        idx for idx, _ in
        sorted(fused.items(), key=lambda x: -x[1])[:config["k"]]
    ]

def retrieve(qid, config, index):
    candidate_indices = first_stage_rank(qid, config, index)

    if config["reranker"] is not None:
        pairs = [
            (queries[qid], index["texts"][idx])
            for idx in candidate_indices
        ]
        scores = get_reranker(config["reranker"]).predict(
            pairs,
            batch_size=8,
            show_progress_bar=False,
        )
        order = np.argsort(-np.asarray(scores))
        candidate_indices = [candidate_indices[i] for i in order]

    candidate_doc_ids = [index["doc_ids"][i] for i in candidate_indices]

    final_doc_ids = []
    for doc_id in candidate_doc_ids:
        if doc_id not in final_doc_ids:
            final_doc_ids.append(doc_id)
        if len(final_doc_ids) == FINAL_K:
            break

    return candidate_doc_ids, final_doc_ids

def evaluate(config, qids):
    index = build_index(config)
    candidate_hits = final_hits = total = 0
    failures = []

    for qid in qids:
        candidate_docs, final_docs = retrieve(qid, config, index)
        gold = qrels[qid]

        candidate_hits += len(gold.intersection(candidate_docs))
        final_hits += len(gold.intersection(final_docs))
        total += len(gold)

        if not gold.intersection(final_docs):
            failures.append({
                "type": (
                    "retrieval_miss"
                    if not gold.intersection(candidate_docs)
                    else "ranking_miss"
                ),
                "query": queries[qid],
                "gold_titles": [corpus[x]["title"] for x in gold],
                "retrieved_titles": [corpus[x]["title"] for x in final_docs],
            })

    return {
        "candidate_recall": candidate_hits / total,
        "recall": final_hits / total,
        "failures": failures[:4],
        "chunk_count": len(index["texts"]),
    }

## 4. DeepSeek 提出下一组配置

DeepSeek 可以选择六类参数，但当前：

- `embedding` 只有一个可用值，因此不会被随意替换；
- `reranker` 可以在关闭和 Qwen3-Reranker 之间切换；
- 其他参数均按原搜索空间选择。

In [ ]:
def config_key(config):
    return json.dumps(config, sort_keys=True)

def validate_config(config, tried):
    if set(config) != set(SEARCH_SPACE):
        raise ValueError("配置字段不完整")

    for name, value in config.items():
        if value not in SEARCH_SPACE[name]:
            raise ValueError(f"{name}={value!r} 不在搜索空间")

    if config_key(config) in tried:
        raise ValueError("该配置已经执行过")

    return config

def parse_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(text)

def propose_next_config(current_config, result, history, tried):
    state = {
        "goal": f"Recall@{FINAL_K} >= {TARGET_RECALL}",
        "current_config": current_config,
        "result": {
            "candidate_recall": round(result["candidate_recall"], 4),
            "recall_at_5": round(result["recall"], 4),
            "chunk_count": result["chunk_count"],
            "failure_examples": result["failures"],
        },
        "previous_trials": history,
        "search_space": SEARCH_SPACE,
    }

    response = client.chat.completions.create(
        model="deepseek-v4-pro",
        messages=[
            {
                "role": "system",
                "content": (
                    "你是 RAG 配置优化器。请根据真实指标和失败案例，"
                    "从给定搜索空间提出下一组完整配置。"
                    "优先形成可验证的单步假设，每轮最多修改两个参数，"
                    "不得返回历史中已经执行过的配置，也不得虚构提升结果。"
                    '只输出 JSON：{"hypothesis":"...", "next_config":{...}}'
                ),
            },
            {
                "role": "user",
                "content": json.dumps(state, ensure_ascii=False),
            },
        ],
        stream=False,
        max_tokens=600,
        reasoning_effort="high",
        extra_body={"thinking": {"type": "enabled"}},
    )

    proposal = parse_json(response.choices[0].message.content)
    proposal["next_config"] = validate_config(
        proposal["next_config"],
        tried,
    )
    return proposal

## 5. 运行 Loop

每轮都使用完整配置进行真实实验；达到目标或达到最大轮数后停止。

In [ ]:
def run_loop():
    config = INITIAL_CONFIG.copy()
    tried, history = set(), []

    for round_id in range(1, MAX_ROUNDS + 1):
        tried.add(config_key(config))
        result = evaluate(config, train_ids)

        row = {
            "round": round_id,
            **config,
            "candidate_recall": round(result["candidate_recall"], 4),
            "recall@5": round(result["recall"], 4),
            "chunks": result["chunk_count"],
        }
        history.append(row)

        print(f"\nRound {round_id}")
        print("Config:", config)
        print(
            f"Candidate recall={result['candidate_recall']:.3f}, "
            f"Recall@{FINAL_K}={result['recall']:.3f}"
        )

        if result["recall"] >= TARGET_RECALL:
            print("Target reached.")
            break

        proposal = propose_next_config(
            config,
            result,
            history,
            tried,
        )
        print("LLM hypothesis:", proposal["hypothesis"])
        config = proposal["next_config"]

    best = max(history, key=lambda x: x["recall@5"])
    best_config = {name: best[name] for name in SEARCH_SPACE}
    return best_config, history

best_config, history = run_loop()
pd.DataFrame(history)

## 6. Held-out 验证

最终配置在 Loop 没有看过的查询上执行相同的完整 Pipeline。

In [ ]:
holdout = evaluate(best_config, holdout_ids)

print("Best config:", best_config)
print(f"Held-out candidate recall: {holdout['candidate_recall']:.3f}")
print(f"Held-out Recall@{FINAL_K}: {holdout['recall']:.3f}")
print(
    "Decision:",
    "PASS" if holdout["recall"] >= TARGET_RECALL else "BLOCK",
)

## 参数含义总结

| 参数 | 本示例中的真实作用 |
|---|---|
| `chunk_size` | 控制每个文本块的字符长度 |
| `chunk_overlap` | 控制相邻文本块的字符重叠 |
| `embedding` | 选择实际执行向量化的 Embedding 模型；当前只有 Qwen3 |
| `k` | 第一阶段保留并送往后续排序的候选 chunk 数 |
| `reranker` | 关闭重排，或使用 Qwen3-Reranker 真实重排 |
| `hybrid` | 关闭时仅 Dense；开启时融合 Dense 与 BM25 |

这里保留的是原文的完整参数框架，而不是只优化 chunk 参数。